# Intyoza: model training and evaluation

**Project:** Intyoza, a sexual and reproductive health (SRH) learning platform for young people in Rwanda (10 to 24 years)
**Language of answers:** Kinyarwanda
**Notebook:** 1 of 4. Trains and evaluates every model the chatbot uses.

**Learning first.** Intyoza is first a place to learn: a young person asks *"Agakingirizo ni iki?"* or *"Imihango ni iki?"* and gets a clear, correct answer in Kinyarwanda, then keeps learning through related questions, topics and the quiz. Safety is a net underneath: help contacts are shown when a message is about danger, without taking answers away from learning questions.

## What this notebook produces

| Model | Role in the chatbot | Method |
|---|---|---|
| Question matcher | Finds the approved question that a message is asking, in Kinyarwanda or English | multilingual-e5-large-instruct (Microsoft, MIT licence) fine-tuned on young people's phrasings with hard negatives |
| Risk models (2) | Detect messages that need urgent help (violence, self-harm, medical emergency) | Logistic regression on matcher embeddings + character n-gram model |
| Safety layer | Combines both risk models and the matched question; tuned to flag at most 5% of normal messages, catching as many dangerous ones as possible | Threshold search on held-out data |
| Intent model | Separates health questions from greetings, thanks and off-topic chat | Logistic regression on matcher embeddings |
| Language model | Identifies Kinyarwanda and English | Character n-gram logistic regression |
| Answer cut-offs (LO, HI) | Decide when to answer directly, ask "did you mean", or not answer | Calibrated on held-out scores |

## Pipeline

1. Environment setup
2. Configuration
3. Load the dataset
4. Preprocessing and data quality
5. Exploratory data analysis
6. Train/test split check
7. Base model choice and baseline
8. Hard-negative mining
9. Fine-tune the question matcher
10. Evaluate the matcher
11. Answer cut-offs
12. Error analysis
13. Risk models
14. Safety calibration
15. Intent and language models
16. Learning-first check and test conversations
17. Save the models and the model card
18. Export (Colab)
19. Optional: refresh the small models only

## How to run

- **Google Colab (recommended, about 20 minutes):** upload the `Intyoza` folder to Google Drive, open this notebook from it, select *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*.
- **Anaconda (offline, CPU, several hours for step 9):** run `python download_model.py` once with internet, then *Run All*.

Outputs are written to `models/` (models, index cache, model card) and to the project folder (`error_report.csv`).

## 1. Environment setup

Detects whether the notebook runs in Google Colab or locally, mounts Google Drive in Colab, moves into the `Intyoza` folder and chooses the base model source:

- **Colab:** the base model is downloaded from Hugging Face.
- **Anaconda:** the base model is read from `base_model/e5-large-instruct` (saved once by `download_model.py`), and all Hugging Face libraries are forced offline.

Random seeds are fixed so results can be reproduced.

In [ ]:
import os, sys, glob, random, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less wasted GPU memory

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working") and not IN_COLAB
ONLINE = IN_COLAB or IN_KAGGLE        # Colab or Kaggle: models are downloaded from the internet
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    hits = glob.glob("/content/drive/MyDrive/**/Intyoza/intyoza_core.py", recursive=True)
    assert hits, "Upload the Intyoza folder to your Google Drive first."
    os.chdir(os.path.dirname(hits[0]))
    !pip -q install -U sentence-transformers datasets accelerate
    BASE_MODEL = "intfloat/multilingual-e5-large-instruct"
elif IN_KAGGLE:      # Kaggle: inputs are read-only; work on a copy in /tmp (only /kaggle/working is saved, kept small)
    import shutil
    hits = glob.glob("/kaggle/input/**/intyoza_core.py", recursive=True)
    assert hits, "Add the Intyoza folder as input: Add Input > Datasets (or Notebook Output) > intyoza."
    def _rank(p):        # prefer the input with the most training done (notebook 4 output > notebook 1 > code only)
        d = os.path.dirname(p)
        return (os.path.isdir(os.path.join(d, "models", "intyoza_writer")), os.path.isdir(os.path.join(d, "models", "intyoza_matcher")))
    src = os.path.dirname(max(hits, key=_rank))
    if not os.path.exists("/tmp/Intyoza/intyoza_core.py"):
        print("Copying", src, "to /tmp/Intyoza ...")
        shutil.copytree(src, "/tmp/Intyoza", dirs_exist_ok=True)
        for out in glob.glob("/kaggle/input/**/intyoza_outputs", recursive=True):   # results of notebook 4
            print("Adding", out)
            shutil.copytree(out, "/tmp/Intyoza", dirs_exist_ok=True)
    for part in ("intyoza_matcher", "intyoza_writer", "doc_index"):   # models uploaded separately (intyoza_models.zip...)
        if not os.path.exists(f"/tmp/Intyoza/models/{part}"):
            found = [d for d in glob.glob(f"/kaggle/input/**/{part}", recursive=True) if os.path.isdir(d)]
            if found:
                models_dir = os.path.dirname(found[0])
                print("Copying models from", models_dir)
                shutil.copytree(models_dir, "/tmp/Intyoza/models", dirs_exist_ok=True)
    os.chdir("/tmp/Intyoza")
    !pip -q install -U sentence-transformers datasets accelerate
    BASE_MODEL = "intfloat/multilingual-e5-large-instruct"
else:
    assert os.path.exists("intyoza_core.py"), "Open this notebook from inside the Intyoza folder."
    BASE_MODEL = os.path.join(os.getcwd(), "base_model", "e5-large-instruct")
    assert os.path.exists(os.path.join(BASE_MODEL, "config.json")), "Run  python download_model.py  once with internet."
    os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1")

import numpy as np, pandas as pd, torch
FOLDER = os.getcwd()
MODELS = os.path.join(FOLDER, "models")
sys.path.insert(0, FOLDER)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
pd.set_option("display.max_colwidth", 90)
%matplotlib inline

import importlib, intyoza_core, evaluation, preprocess
for m in (preprocess, intyoza_core, evaluation):
    importlib.reload(m)
from intyoza_core import *
import evaluation as ev

print("Project folder:", FOLDER)
print("Environment:   ", "Google Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local (Anaconda)"))
print("Device:        ", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (matcher training will take several hours)")
print("Libraries:     ", ev.library_versions())

## 2. Configuration

All settings that affect training are defined here and recorded in the model card, so every trained model can be traced back to its settings.

| Setting | Meaning |
|---|---|
| `epochs`, `batch_size`, `learning_rate`, `warmup_ratio` | Fine-tuning schedule for the matcher |
| `mini_batch_size` | The batch is processed in chunks of this size to fit the T4's memory; the result is the same as the full batch |
| `max_seq_length` | Longest message the matcher reads (tokens); young people's messages are short |
| `hard_negatives` | Look-alike questions used as negatives for each question |
| `safety_max_false_alarm` | Largest share of normal held-out messages the safety layer may flag as risky (learning first) |
| `risk_C` | Regularisation of the risk and intent classifiers (higher = fits training data more closely) |

In [ ]:
CONFIG = {
    "base_model": "intfloat/multilingual-e5-large-instruct",
    "instruction": MATCH_PROMPT.strip(),
    "epochs": 4,
    "batch_size": 32,
    "mini_batch_size": 8,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.1,
    "max_seq_length": 128,
    "loss": "MultipleNegativesRankingLoss",
    "batch_sampler": "NO_DUPLICATES",
    "hard_negatives": 3,
    "safety_max_false_alarm": 0.05,
    "risk_C": 4,
    "seed": SEED,
    "device": DEVICE,
}
pd.Series(CONFIG, name="value").to_frame()

## 3. Load the dataset

**Sources**

- `intyoza_data.json`: the approved questions and answers (English and Kinyarwanda, with age-specific versions), referral information, help messages, learning paths, and the phrasings young people might use.
- `knowledge/Intyoza_Dataset.xlsx`: the review workbook. Edits made there (changed answers, new questions, new phrasings) are applied on top of the JSON, so the team can improve the data without touching code.

**Records**

- *Question:* `id`, `topic`, `risk` (None, Low, High, Emergency), question and answer in both languages, referral, sources.
- *Phrasing:* a message as a young person might write it, linked to a question (`qa_id`) or only to a risk level or intent, with a language and a `train` / `test` split.

In [ ]:
data, qa, raw_examples, load_info = load_raw(FOLDER)
print(f"{len(qa)} approved questions, {len(raw_examples)} phrasings loaded")
pd.Series(load_info, name="value").to_frame()

In [ ]:
# One approved question, as stored
sample = qa["CON-001"]
pd.DataFrame({"English": [sample["q_en"], sample["answers"]["default"]["en"]],
              "Kinyarwanda": [sample["q_rw"], sample["answers"]["default"]["rw"]]},
             index=["question", "answer"])

## 4. Preprocessing and data quality

The same normalisation is applied to training data and to every live chat message (`preprocess.normalize`), so the model sees text in one consistent form:

- Unicode normalisation (NFKC) and one apostrophe style
- Lower case; links removed; letters repeated three or more times reduced ("pleaseeee" becomes "pleasee"); repeated punctuation reduced
- Texting shortcuts and spelling variants mapped to one form (for example *plz* to *please*, *ntago* to *ntabwo*, *kondomu* to *agakingirizo*)

Then the phrasings are checked (`preprocess.clean_examples`):

- empty or too-short phrasings are removed
- exact duplicates are removed
- the same text filed under two different questions is reported (a labelling conflict)
- a test phrasing identical to a training phrasing is removed (it would inflate accuracy)
- every question gets at least one held-out test phrasing when it has enough phrasings

In [ ]:
from preprocess import normalize, clean_examples
examples, prep_report = clean_examples(raw_examples, verbose=False)

report = pd.Series(dict(prep_report), name="phrasings").to_frame()
report.loc["loaded"] = len(raw_examples)
report

In [ ]:
# What normalisation changes: a sample of phrasings before and after
changed = [(e["raw"], e["text"]) for e in examples if e["raw"].strip().lower() != e["text"]]
print(f"{len(changed)} of {len(examples)} phrasings were changed by normalisation. Sample:")
pd.DataFrame(random.Random(SEED).sample(changed, min(12, len(changed))), columns=["before", "after"])

## 5. Exploratory data analysis

Checks that the dataset is balanced enough to train on: how questions are spread across topics and risk levels, how many phrasings each language has, and which questions have too few phrasings (these are the most likely to be matched wrongly).

In [ ]:
overview = ev.dataset_overview(qa, examples)
overview["questions_by_topic_and_risk"]

In [ ]:
display(overview["phrasings_by_language_and_split"])
display(overview["phrasings_by_kind"])
display(overview["translation_coverage"])

In [ ]:
ev.plot_dataset(qa, examples)

In [ ]:
print("Training phrasings per question:")
display(overview["training_phrasings_per_question"].T)
thin = overview["thin_questions"]
print(f"{len(thin)} questions have fewer than 5 training phrasings (add phrasings for these in the workbook):")
thin.assign(question=[qa[i]["q_en"][:70] for i in thin.index]).head(20)

## 6. Train/test split check

Every metric in this notebook is measured on **held-out test phrasings** that the models never see during training. This step verifies the split is clean: no text appears in both sets, and both languages are represented in the test set.

In [ ]:
train_texts = {e["text"] for e in examples if e["split"] == "train"}
test_set = [e for e in examples if e["split"] == "test"]
leaks = [e["text"] for e in test_set if e["text"] in train_texts]
assert not leaks, f"{len(leaks)} test phrasings also appear in training: {leaks[:5]}"

split = pd.DataFrame([{"split": e["split"], "language": ev.LANG_NAME.get(e["lang"], e["lang"]),
                       "kind": "question" if e["qa_id"] else e["intent"]} for e in examples])
print("No overlap between training and test phrasings.")
pd.crosstab([split["split"], split["language"]], split["kind"], margins=True, margins_name="Total")

## 7. Base model choice and baseline

**Choice: multilingual-e5-large-instruct** (Microsoft; MIT licence; Wang et al., 2024, *Multilingual E5 Text Embeddings: A Technical Report*, arXiv:2402.05672).

| Criterion | Evidence |
|---|---|
| Understands Kinyarwanda | Independent Kinyarwanda benchmark (KinyaEmbed paper, arXiv:2608.26941): sentence similarity 0.60 and 0.55 (Spearman), retrieval P@1 0.98, against 0.45, 0.22 and 0.59 for LaBSE |
| Clear licence | MIT: free to use, change and deploy, including in a public platform |
| Documented training data | Described in the technical report (multilingual weakly supervised pairs, then supervised fine-tuning) |
| Fits the hardware | 560M parameters: trains on a free T4 GPU, runs on a laptop CPU |
| Serves both jobs | Matching questions (similarity) and finding document passages (retrieval) with one model |

The model reads every message with a short instruction in front (*"Given a young person's question about sexual and reproductive health, retrieve the question with the same meaning"*). The instruction is saved with the model, so chat uses it automatically.

**Baseline on our own data.** Before training, both the chosen model and LaBSE (the previous choice) are measured on the held-out phrasings. This shows on Intyoza's own Kinyarwanda data which base is stronger, and gives the baseline that fine-tuning must beat.

**Metrics**

- **Top-1:** the first match is the right question (the bot answers correctly).
- **Top-3:** the right question is among the first three.
- **MRR (mean reciprocal rank):** 1 when the right question is first, 1/2 when second, and so on, averaged.

In [ ]:
from sentence_transformers import SentenceTransformer
base_model = set_prompts(SentenceTransformer(BASE_MODEL, device=DEVICE)).float()   # 32-bit weights for training
base_model.max_seq_length = CONFIG["max_seq_length"]

baseline_index = Index(base_model, qa, examples)
baseline_ranked = ev.rank_test_set(baseline_index, examples, qa)
baseline_metrics = ev.retrieval_metrics(baseline_ranked)
baselines = {"multilingual-e5-large-instruct (chosen)": baseline_metrics}

COMPARE_WITH_LABSE = True        # set False to skip (saves about 2 minutes)
labse_source = "sentence-transformers/LaBSE" if ONLINE else os.path.join(FOLDER, "base_model", "LaBSE")
if COMPARE_WITH_LABSE and (ONLINE or os.path.exists(os.path.join(labse_source, "config.json"))):
    labse = SentenceTransformer(labse_source, device=DEVICE)
    labse.max_seq_length = CONFIG["max_seq_length"]
    labse_ranked = ev.rank_test_set(Index(labse, qa, examples), examples, qa)
    baselines["LaBSE (previous)"] = ev.retrieval_metrics(labse_ranked)
    by_lang_labse = ev.metrics_by(labse_ranked, "lang").assign(model="LaBSE")
    del labse
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
else:
    by_lang_labse = None

print("Before fine-tuning, on held-out phrasings:")
display(pd.DataFrame(baselines).T[["n", "top1", "top3", "mrr"]])
by_lang = ev.metrics_by(baseline_ranked, "lang").assign(model="multilingual-e5-large-instruct")
pd.concat([by_lang, by_lang_labse]) if by_lang_labse is not None else by_lang

## 8. Hard-negative mining

Many SRH questions look alike: *"Can the emergency pill cause abortion?"* and *"Where can I get a safe abortion?"* share most of their words. Training only on random negatives does not teach the model to separate them.

For each approved question, the base model finds the **3 most similar other questions**. During training, each phrasing is paired with its correct question (positive) and one of these look-alikes (hard negative), taking turns so every look-alike is used.

In [ ]:
negs = hard_negatives(base_model, qa, n=CONFIG["hard_negatives"])
pd.DataFrame([{"question": qa[q]["q_en"][:55], "look-alike 1": qa[n[0]]["q_en"][:45], "look-alike 2": qa[n[1]]["q_en"][:45]}
              for q, n in list(negs.items()) if q in ("CON-011", "CON-008", "PRG-002", "HIV-005", "MEN-001", "GBV-001")])

In [ ]:
triplets = matcher_triplets(qa, examples, negs)
print(f"{len(triplets)} training triplets (phrasing, correct question, look-alike question). Example:")
a, p, n = triplets[0]
pd.DataFrame({"text": [a, p[:150] + "...", n[:150] + "..."]}, index=["phrasing", "positive", "hard negative"])

## 9. Fine-tune the question matcher

**Loss:** Multiple Negatives Ranking Loss. For each phrasing in a batch, its own question must score higher than every other question in the batch (in-batch negatives) and higher than its hard negative.

**Memory:** the loss is the *cached* version (GradCache, Gao et al. 2021): each batch of 32 is computed in chunks of 8, which gives the same result as the full batch while fitting the T4's 15 GB.

**Batch sampler:** `NO_DUPLICATES` prevents the same question from appearing twice in a batch, which would otherwise be treated as a wrong answer.

The same instruction is given to every text during training, matching how the model is used in chat.

On a T4 GPU this takes about 12 minutes; on a laptop CPU, several hours.

In [ ]:
import math
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
try:                                    # sentence-transformers 6 and later
    from sentence_transformers.sentence_transformer import losses
    from sentence_transformers.sentence_transformer.training_args import BatchSamplers
except ImportError:                     # sentence-transformers 3 to 5
    from sentence_transformers import losses
    from sentence_transformers.training_args import BatchSamplers

model = base_model.float()  # fine-tuned in place; 32-bit weights (16-bit weights cannot be trained with fp16 mixed precision)
train_ds = Dataset.from_dict({"anchor": [a for a, _, _ in triplets],
                              "positive": [p for _, p, _ in triplets],
                              "negative": [n for _, _, n in triplets]})
total_steps = math.ceil(len(train_ds) / CONFIG["batch_size"]) * CONFIG["epochs"]
warmup_steps = int(CONFIG["warmup_ratio"] * total_steps)     # learning rate rises during the first 10% of steps
print(f"{len(train_ds)} triplets, {total_steps} training steps, {warmup_steps} warm-up steps")
args = dict(
    output_dir=os.path.join(FOLDER, "matcher-checkpoints"),
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=warmup_steps,
    fp16=(DEVICE == "cuda"),
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="no",
    logging_steps=20,
    report_to="none",
    seed=SEED,
)
if base_model.prompts:                   # the instruction goes in front of every training text
    args["prompts"] = MATCH_PROMPT
args = SentenceTransformerTrainingArguments(**args)
if DEVICE == "cuda":
    torch.cuda.empty_cache()
loss = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=CONFIG["mini_batch_size"])
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_ds, loss=loss)
train_result = trainer.train()
print(f"Training finished in {train_result.metrics['train_runtime'] / 60:.1f} minutes, "
      f"final loss {train_result.metrics['train_loss']:.4f}")

In [ ]:
ev.plot_training_loss(trainer.state.log_history)

## 10. Evaluate the matcher

The question index represents each approved question by the average embedding of its question text, its answer and its training phrasings. Held-out phrasings are ranked against it.

The comparison with the baseline shows what fine-tuning added. Results are broken down by language, since Kinyarwanda is the main language of the product, and by topic, to show where more phrasings are needed.

In [ ]:
index = Index(model, qa, examples, cache=os.path.join(MODELS, INDEX_CACHE))
ranked = ev.rank_test_set(index, examples, qa)
fine_metrics = ev.retrieval_metrics(ranked)
ev.compare({"Before fine-tuning": baseline_metrics, "Fine-tuned": fine_metrics})

In [ ]:
by_language = ev.metrics_by(ranked, "lang")
by_language

In [ ]:
by_topic = ev.metrics_by(ranked, "topic")
print("Topics from weakest to strongest (top-1 accuracy):")
by_topic

## 11. Answer cut-offs

The top match score decides what the bot does:

| Score | Action |
|---|---|
| at least **HI** | answer directly with the approved answer |
| between **LO** and **HI** | ask "did you mean" with the three closest questions |
| below **LO** | do not answer from the approved list (offer topics, or search the documents) |

**HI** is set at the 90th percentile of scores for wrong matches and off-topic messages, so wrong answers given directly are rare. **LO** is set at the median score of off-topic messages.

In [ ]:
LO, HI = tune_thresholds(index, examples)
print(f"LO = {LO}   HI = {HI}")
ev.plot_scores(ranked, LO, HI)

In [ ]:
print("What the bot would do with each held-out message:")
ev.decision_outcomes(ranked, LO, HI)

## 12. Error analysis

The most frequent confusions show which questions need clearer answers or more distinctive phrasings. Every mistake is saved to `error_report.csv`; add phrasings for those questions in `knowledge/Intyoza_Dataset.xlsx` and retrain (or run step 19).

In [ ]:
ev.confused_pairs(ranked, qa)

In [ ]:
errors = ev.mistakes(ranked, qa)
errors.to_csv(os.path.join(FOLDER, "error_report.csv"), index=False)
print(f"{len(errors)} mistakes saved to error_report.csv ({errors['right_answer_in_top3'].mean():.0%} of them "
      f"still offer the right question in 'did you mean'). Sample:")
errors.head(10)

## 13. Risk models

Two complementary risk classifiers are trained on the four risk levels (None, Low, High, Emergency):

- **Embedding model:** logistic regression on matcher embeddings; understands meaning across languages.
- **Character model:** TF-IDF on character n-grams (2 to 5) with logistic regression; catches spelling variants and words the embedding smooths over (for example *kwiyahura*, *gufatwa ku ngufu*).

Class weights are balanced, because risky messages are a minority. Each model is evaluated alone here; step 14 combines them.

In [ ]:
risk_clf, intent_clf = train_heads(model, examples, C=CONFIG["risk_C"])
risk_char = train_char_risk(examples, C=CONFIG["risk_C"])

y_true, y_pred_emb = ev.risk_predictions_single(risk_clf, model, examples)
test_texts = [e["text"] for e in examples if e["split"] == "test"]
y_pred_char = list(risk_char.predict(test_texts))

print("Embedding risk model")
display(ev.classification_table(y_true, y_pred_emb, RISKS))
print("Character risk model")
display(ev.classification_table(y_true, y_pred_char, RISKS))

## 14. Safety calibration

Intyoza is a learning platform, so a normal question must not be answered with a crisis message. The safety layer is therefore tuned to flag **at most 5% of normal held-out messages** as risky, and among the settings that respect this limit, the one that catches the **most** dangerous messages (High and Emergency) is chosen.

The safety layer flags a message as risky when either risk model's probability of High or Emergency passes a threshold. Optionally it also inherits the risk of a clearly matched urgent question. Every threshold from 0.15 to 0.95 is tried, with and without the matched-question rule.

In chat, the safety decision is applied in this order:

| Message | What the young person gets |
|---|---|
| Clear match to a learning question | **The answer, always.** If the message also looks risky, help contacts are added: a short line under the answer, or the full help message above it when the danger is very clear (for example *"I was raped, can I get pregnant?"*) |
| Clear greeting, thanks or small talk | A friendly reply and the learning menu, unless the risk signal is strong |
| Dangerous message that matches an urgent question (rape, self-harm, heavy bleeding) | That answer, with the right help contacts |
| Other dangerous message | The help message (violence, self-harm or medical, chosen from the closest urgent question); then supportive follow-up instead of repeating the list |
| Message with very clear danger words (*kwiyahura*, *raped*, *kill myself*) | Always treated as an emergency |

In [ ]:
safety = tune_safety(index, risk_clf, risk_char, model, examples, qa, LO,
                     max_false_alarm=CONFIG["safety_max_false_alarm"], verbose=False)
print(f"Chosen threshold {safety.threshold}, use matched question's risk = {safety.inherit}")
print(f"Normal messages flagged: {safety.false_alarm_rate:.1%} (limit {safety.max_false_alarm:.0%})   "
      f"Dangerous messages caught: {safety.recall:.1%}")
safety.table

In [ ]:
ev.plot_safety(safety.table, safety.max_false_alarm, (safety.recall, safety.false_alarm_rate))

In [ ]:
y_true_s, y_pred_s, te_s = ev.risk_predictions_safety(safety, index, model, examples, qa, LO)
ev.plot_confusion(ev.confusion(y_true_s, y_pred_s, RISKS), "Safety layer on held-out messages")
misses = ev.risky_misses(y_true_s, y_pred_s, te_s)
print(f"Dangerous messages the safety layer did not flag: {len(misses)}. Add phrasings like these to the workbook, "
      "so the next training run learns them:")
misses

## 15. Intent and language models

- **Intent** separates health questions from greetings, thanks/goodbye and off-topic chat, so small talk gets a friendly reply and the topic menu instead of a wrong health answer.
- **Language** identifies Kinyarwanda and English. Replies are always in Kinyarwanda by default; the language model is used in bilingual mode (`chat.py --bilingual`) and to keep the conversation language for one-word messages.

In [ ]:
lang_model = train_lang_model(examples)
test = [e for e in examples if e["split"] == "test"]
X_test = model.encode([e["text"] for e in test], normalize_embeddings=True)

print("Intent")
display(ev.classification_table([e["intent"] for e in test], list(intent_clf.predict(X_test)), INTENTS))
print("Language")
display(ev.classification_table([e["lang"] for e in test], list(lang_model.predict([e["text"] for e in test])), ["rw", "en"]))

## 16. Learning-first check and test conversations

The trained parts are assembled into the chatbot exactly as in the app. Every held-out phrasing is then sent through the full chatbot, each in a fresh conversation, to check the two promises of the platform:

- **Learning questions** (risk None or Low) get their answer, directly or in "did you mean", and are almost never answered with a crisis message instead.
- **Dangerous messages** (High or Emergency) get help contacts.

Then a set of typical messages shows the replies themselves.

In [ ]:
heads = {"risk": risk_clf, "intent": intent_clf, "lang": lang_model, "lo": LO, "hi": HI, "safety": safety}
bot = build_bot(data, qa, examples, model, heads, index=index)

learning_summary, sent_to_help, learning_metrics = ev.learning_first_report(bot, examples, qa)
display(pd.Series(learning_metrics, name="share").to_frame())
display(learning_summary)
print(f"Learning questions answered with a crisis message instead of their answer: {len(sent_to_help)}")
sent_to_help.head(15)

In [ ]:
conversation = [
    "mwaramutse",
    "imihango imara iminsi ingahe?",
    "ese agakingirizo karinda inda n'indwara?",
    "ni ibisanzwe ko umuhungu arota akasohora?",
    "nakoze imibonano ejo ntitwakoresheje agakingirizo, nakora iki?",
    "agakingirizo ni iki?",
    "imihango ni iki?",
    "ndumva nshaka kwiyahura",
    "ese ikipe ya APR yatsinze ejo?",
    "murakoze",
]
rows = []
for msg in conversation:
    bot.reset()                          # each message as the start of a new conversation
    r = bot.reply(msg)
    rows.append({"message": msg, "response type": r["type"], "help": r.get("help", ""), "reply": r["text"][:160].replace("\n", " ") + "...",
                 "options offered": len(r.get("options", []))})
pd.DataFrame(rows)

## 17. Save the models and the model card

Saved to `models/`:

| File | Content |
|---|---|
| `intyoza_matcher/` | fine-tuned sentence encoder |
| `intyoza_heads.joblib` | risk, intent and language models, safety layer, LO/HI cut-offs |
| `index_cache.npz` | question index vectors (chat starts in seconds) |
| `model_card.json`, `MODEL_CARD.md` | data, settings, metrics and limitations of this training run |

In [ ]:
save_all(MODELS, model, risk_clf, intent_clf, lang_model, LO, HI, safety)

data_stats = {"fingerprint": ev.data_fingerprint(qa, examples), "questions": len(qa),
              "topics": len({q["topic"] for q in qa.values()}), "phrasings": len(examples),
              "train": sum(e["split"] == "train" for e in examples), "test": sum(e["split"] == "test" for e in examples),
              "kinyarwanda": sum(e["lang"] == "rw" for e in examples), **load_info}
metrics = {"baseline": baseline_metrics, "fine_tuned": fine_metrics,
           "by_language": {r["lang"]: {k: r[k] for k in ("n", "top1", "top3", "mrr")} for _, r in by_language.iterrows()},
           "lo": LO, "hi": HI,
           "safety": {"threshold": safety.threshold, "use_matched_risk": safety.inherit, "recall": safety.recall,
                      "false_alarm_rate": safety.false_alarm_rate, "max_false_alarm": safety.max_false_alarm},
           "learning_first": learning_metrics,
           "training": {k: round(v, 4) if isinstance(v, float) else v for k, v in train_result.metrics.items()}}
ev.write_model_card(MODELS, CONFIG, data_stats, metrics)
print("Saved to", MODELS)
for f in sorted(os.listdir(MODELS)):
    print("  ", f)

## 18. Export (Colab)

In Colab, the `models` folder is zipped into `intyoza_models.zip` in the project folder on Drive. Download it and unzip it inside `Intyoza` on the laptop to chat offline (`start_intyoza.bat` or `python chat.py`).

In [ ]:
if ONLINE:
    import shutil
    shutil.make_archive(os.path.join("/kaggle/working" if IN_KAGGLE else FOLDER, "intyoza_models"), "zip", FOLDER, "models")
    print("Created intyoza_models.zip in", "/kaggle/working (Output tab)" if IN_KAGGLE else FOLDER)
else:
    print("Local run: models are already in", MODELS, "- chat with  python chat.py")

## 19. Optional: refresh the small models only

After small edits to the workbook (new phrasings, corrected answers), the matcher does not need retraining. This step keeps `models/intyoza_matcher`, retrains the risk, intent and language models and the safety layer, and saves them (a few minutes).

To use it: restart the runtime, run step 1, set `REFRESH = True` and run this cell.

In [ ]:
REFRESH = False
if REFRESH:
    importlib.reload(intyoza_core)
    intyoza_core.refresh_small_models(FOLDER)